# Descriptores de forma desde cero, con NumPy

Visión Artificial · CUTLAJO

Es el mismo recorrido que `formas_paso_a_paso.ipynb`, pero **sin OpenCV ni scikit-image**: cada operación la escribimos nosotros con NumPy, para ver qué hay dentro de cada función que normalmente usamos como caja negra.

```
original → gris → binaria → morfología → etiquetas → contorno → descriptores → tabla (pandas)
```

- **NumPy**: una imagen es una matriz, así que todo el procesamiento es álgebra de arreglos.
- **pandas**: solo al final, para la tabla de descriptores (eso sí son datos tabulares).
- **Pillow**: solo para *leer* el JPG. Decodificar JPG a mano sería otro curso.

Funciona igual en local y en Colab (Archivo → Subir notebook).

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.rcParams["image.cmap"] = "gray"


def mostrar(*imagenes, titulos=None, tam=4):
    fig, ejes = plt.subplots(1, len(imagenes), figsize=(tam * len(imagenes), tam))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(ejes), imagenes)):
        ax.imshow(im, vmin=0, vmax=1 if im.dtype == bool else 255, interpolation="nearest")
        ax.set_title(titulos[i] if titulos else "")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 0. Conseguir la imagen

En Colab no existe `formas.jpg`: la celda te pide subirla. Si cancelas o no estás en Colab y no está el archivo, genera una imagen de ejemplo con las mismas tres figuras.

In [ ]:
from pathlib import Path

RUTA = Path("formas.jpg")
if not RUTA.exists():
    try:
        from google.colab import files
        print("Sube formas.jpg (o cancela para usar una imagen de ejemplo)")
        subidos = files.upload()
        if subidos:
            RUTA = Path(next(iter(subidos)))
    except Exception:
        pass

if not RUTA.exists():
    from PIL import Image, ImageDraw
    im = Image.new("RGB", (640, 480), (230, 230, 230))
    d = ImageDraw.Draw(im)
    d.ellipse((60, 80, 260, 280), fill=(220, 30, 32))
    d.rectangle((360, 160, 580, 400), fill=(31, 60, 200))
    d.polygon([(50, 455), (290, 455), (170, 305)], fill=(40, 170, 60))
    im.save(RUTA, quality=95)
    print("Generé una imagen de ejemplo:", RUTA)
print("Usando:", RUTA)

## 1. La imagen es una matriz

`np.asarray` convierte la imagen en un arreglo de `alto × ancho × 3`. Ojo: Pillow usa **RGB**; OpenCV usaba BGR.

In [ ]:
imagen = np.asarray(Image.open(RUTA).convert("RGB"))
print("forma:", imagen.shape, " tipo:", imagen.dtype)
print("fondo  [10, 10]  =", imagen[10, 10])
print("círculo[180,160] =", imagen[180, 160])
mostrar(imagen, titulos=["Original"], tam=6)

## 2. Escala de grises: un producto punto

$$ gris = 0.299\,R + 0.587\,G + 0.114\,B $$

Con NumPy es un producto matricial entre cada píxel (3 valores) y el vector de pesos. El histograma es contar cuántas veces aparece cada valor: `np.bincount`.

In [ ]:
pesos = np.array([0.299, 0.587, 0.114])
gris = (imagen @ pesos).round().astype(np.uint8)      # (480, 640, 3) @ (3,) -> (480, 640)
histograma = np.bincount(gris.ravel(), minlength=256)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.imshow(gris, vmin=0, vmax=255); a1.set_title("Gris"); a1.axis("off")
a2.bar(np.arange(256), histograma, width=1, color="k"); a2.set_yscale("log")
a2.set_title("Histograma (np.bincount)"); a2.set_xlabel("nivel de gris")
plt.show()

## 3. Binarización: una comparación

En NumPy umbralizar es solo `gris <= T`: el resultado es una matriz de **booleanos** (`True` = figura). No hace falta usar 0 y 255.

In [ ]:
umbrales = [60, 100, 150, 235]
mostrar(*[gris <= t for t in umbrales], titulos=[f"gris <= {t}" for t in umbrales], tam=3.5)

### Otsu, vectorizado

Para cada umbral posible *t* separamos los píxeles en dos clases y medimos qué tan separadas quedan sus medias (**varianza entre clases**):

$$ \sigma_B^2(t) = \frac{\big(\mu_T\,\omega(t) - \mu(t)\big)^2}{\omega(t)\,\big(1-\omega(t)\big)} $$

- $\omega(t)$: fracción de píxeles con valor ≤ t  → `np.cumsum(p)`
- $\mu(t)$: suma acumulada de valor × probabilidad → `np.cumsum(p * niveles)`
- $\mu_T$: media global

Otsu elige el *t* que maximiza $\sigma_B^2$. Los 256 umbrales se evalúan **de una sola vez**, sin `for`.

In [ ]:
p = histograma / histograma.sum()
niveles = np.arange(256)
omega = np.cumsum(p)
mu = np.cumsum(p * niveles)
mu_total = mu[-1]

with np.errstate(divide="ignore", invalid="ignore"):
    varianza_entre = (mu_total * omega - mu) ** 2 / (omega * (1 - omega))
T = int(np.nanargmax(varianza_entre))
print("Umbral de Otsu:", T)

binaria = gris <= T

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(niveles, varianza_entre, "k"); a1.axvline(T, color="r", label=f"T = {T}"); a1.legend()
a1.set_title("Varianza entre clases"); a1.set_xlabel("t")
a2.imshow(binaria); a2.set_title("Binaria"); a2.axis("off")
plt.show()

## 4. Morfología con desplazamientos

El kernel es una matriz pequeña de booleanos. La idea clave: **desplazar** la imagen una vez por cada `True` del kernel y combinar las copias.

- **Erosión** = AND de todas las copias: un píxel sobrevive solo si *todos* sus vecinos (según el kernel) son figura.
- **Dilatación** = OR de todas las copias: un píxel se enciende si *algún* vecino es figura.

(Estrictamente la dilatación usa el kernel reflejado; como el nuestro es simétrico, da lo mismo.)

In [ ]:
kernel = np.array([[0, 0, 1, 0, 0],
                   [1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1],
                   [1, 1, 1, 1, 1],
                   [0, 0, 1, 0, 0]], bool)   # elipse 5x5, igual que la de OpenCV


def _copias_desplazadas(img, k):
    r = k.shape[0] // 2
    pad = np.pad(img, r, constant_values=False)
    alto, ancho = img.shape
    for dy, dx in zip(*np.nonzero(k)):
        yield pad[dy:dy + alto, dx:dx + ancho]


def erosion(img, k):
    return np.logical_and.reduce(list(_copias_desplazadas(img, k)))


def dilatacion(img, k):
    return np.logical_or.reduce(list(_copias_desplazadas(img, k)))


def apertura(img, k):
    return dilatacion(erosion(img, k), k)


def cierre(img, k):
    return erosion(dilatacion(img, k), k)


rng = np.random.default_rng(0)
sucia = binaria.copy()
sucia[rng.random(sucia.shape) < 0.01] = True    # sal
sucia[rng.random(sucia.shape) < 0.01] = False   # pimienta

mostrar(sucia, erosion(sucia, kernel), dilatacion(sucia, kernel),
        titulos=["Con ruido", "Erosión (AND)", "Dilatación (OR)"])
limpia = cierre(apertura(sucia, kernel), kernel)
mostrar(sucia, apertura(sucia, kernel), limpia,
        titulos=["Con ruido", "Apertura", "Apertura + cierre"])

## 5. Etiquetado con búsqueda en anchura (BFS)

Recorremos los píxeles blancos. Cuando encontramos uno sin etiqueta, le damos un número nuevo y "inundamos" todos sus vecinos conectados con una **cola** (`deque`). Es el algoritmo de la cubeta de pintura de Paint.

Usamos **conectividad 8**: los vecinos en diagonal también cuentan.

Este paso sí lleva un `for` de Python píxel por píxel, así que es mucho más lento que la versión en C. Por eso en la práctica se usa `skimage.measure.label` o `cv2.connectedComponents`, que hacen lo mismo en C.

In [ ]:
VECINOS_8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


def etiquetar(binaria):
    alto, ancho = binaria.shape
    fondo = binaria.tolist()            # listas de Python: mucho más rápidas que indexar NumPy en un bucle
    etiquetas = np.zeros((alto, ancho), np.int32)
    vistas = [[0] * ancho for _ in range(alto)]
    actual = 0
    for y, x in zip(*np.nonzero(binaria)):
        if vistas[y][x]:
            continue
        actual += 1
        vistas[y][x] = actual
        cola = deque([(y, x)])
        while cola:
            cy, cx = cola.popleft()
            for dy, dx in VECINOS_8:
                ny, nx = cy + dy, cx + dx
                if 0 <= ny < alto and 0 <= nx < ancho and fondo[ny][nx] and not vistas[ny][nx]:
                    vistas[ny][nx] = actual
                    cola.append((ny, nx))
    etiquetas[:] = vistas
    return etiquetas, actual


etiquetas, n = etiquetar(binaria)
print("Regiones:", n, "  etiquetas:", np.unique(etiquetas))
mostrar(etiquetas * (255 // n), titulos=["Etiquetas"], tam=6)

## 6. Propiedades básicas: área, centroide y caja

Con la máscara de cada región (`etiquetas == i`) casi todo sale de `np.nonzero`:

- **área** = número de píxeles `True` → `mascara.sum()`
- **centroide** = promedio de las coordenadas
- **caja** = mínimo y máximo de las coordenadas

In [ ]:
regiones = []
for i in range(1, n + 1):
    mascara = etiquetas == i
    ys, xs = np.nonzero(mascara)
    regiones.append({
        "id": i,
        "mascara": mascara,
        "ys": ys, "xs": xs,
        "area": mascara.sum(),
        "centroide": (xs.mean(), ys.mean()),
        "bbox": (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
    })
    print(f"Región {i}: área={regiones[-1]['area']}  centroide=({xs.mean():.0f}, {ys.mean():.0f})  "
          f"bbox={regiones[-1]['bbox']}")

## 7. El contorno: trazado de Moore

Para el perímetro necesitamos **recorrer el borde** en orden. El algoritmo de Moore es como caminar con la mano izquierda pegada a la pared:

1. Empezamos en el píxel de más arriba a la izquierda (su vecino de la izquierda seguro es fondo).
2. Desde el último vecino de fondo, giramos en sentido horario alrededor del píxel actual hasta encontrar el siguiente píxel de figura. Ese es el siguiente punto del borde.
3. Repetimos hasta volver al inicio entrando de la misma forma.

El **perímetro** es la suma de los pasos: 1 si el paso es recto, √2 si es diagonal.

In [ ]:
# 8 direcciones (dy, dx) en sentido horario, empezando por el Oeste
DIRS = [(0, -1), (-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1)]


def trazar_contorno(mascara):
    """Regresa los puntos del borde en orden, como arreglo (N, 2) de (x, y)."""
    m = np.pad(mascara, 1)
    ys, xs = np.nonzero(m)
    inicio = p = (int(ys[0]), int(xs[0]))
    b = 0                                   # dirección hacia el último vecino de fondo
    contorno, primer_paso = [p], None
    while True:
        for i in range(1, 9):
            d = (b + i) % 8
            q = (p[0] + DIRS[d][0], p[1] + DIRS[d][1])
            if m[q]:
                break
        else:
            break                           # píxel aislado
        if p == inicio:
            if primer_paso is None:
                primer_paso = d
            elif d == primer_paso:
                break                       # dimos la vuelta completa
        c = (p[0] + DIRS[(d - 1) % 8][0], p[1] + DIRS[(d - 1) % 8][1])
        b = DIRS.index((c[0] - q[0], c[1] - q[1]))
        p = q
        contorno.append(p)
    pts = np.array(contorno[:-1]) - 1       # quitar el padding
    return pts[:, ::-1]                     # (y, x) -> (x, y)


def perimetro(puntos):
    pasos = np.diff(np.vstack([puntos, puntos[:1]]), axis=0)   # cerrar el polígono
    return np.hypot(pasos[:, 0], pasos[:, 1]).sum()


for r in regiones:
    r["contorno"] = trazar_contorno(r["mascara"])
    r["perimetro"] = perimetro(r["contorno"])
    print(f"Región {r['id']}: {len(r['contorno'])} puntos de borde, perímetro = {r['perimetro']:.1f}")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(imagen)
for r in regiones:
    c = r["contorno"]
    ax.plot(*np.vstack([c, c[:1]]).T, "k", lw=2)
    ax.text(*r["centroide"], f"A = {r['area']}\nP = {r['perimetro']:.0f}",
            ha="center", va="center", color="w", fontsize=11, weight="bold")
ax.set_title("Contornos trazados con Moore"); ax.axis("off")
plt.show()

## 8. Circularidad y extensión

$$ circularidad = \frac{4\pi \cdot área}{perímetro^2} \qquad extensión = \frac{área}{área\ de\ la\ caja} $$

| Figura | circularidad | extensión |
|---|---|---|
| Círculo | 1 | π/4 ≈ 0.785 |
| Cuadrado | π/4 ≈ 0.785 | 1 |
| Triángulo equilátero | ≈ 0.605 | 0.5 |

Nuestro triángulo no es equilátero (es más ancho que alto), así que su circularidad sale un poco más baja.

In [ ]:
for r in regiones:
    x0, y0, x1, y1 = r["bbox"]
    r["circularidad"] = 4 * np.pi * r["area"] / r["perimetro"] ** 2
    r["extension"] = r["area"] / ((x1 - x0 + 1) * (y1 - y0 + 1))
    print(f"Región {r['id']}: circularidad = {r['circularidad']:.3f}   extensión = {r['extension']:.3f}")

## 9. Casco convexo: la "liga"

Usamos el algoritmo de **cadena monótona** (Andrew): ordenamos los puntos por *x* y construimos la parte de abajo y la de arriba del casco. Cada vez que los últimos tres puntos giran "hacia adentro" (producto cruz ≤ 0), sacamos el del medio.

Con el casco calculamos:

$$ solidez = \frac{área\ del\ contorno}{área\ del\ casco} \qquad convexidad = \frac{perímetro\ del\ casco}{perímetro} $$

Las áreas de polígonos salen con la **fórmula del zapato** (*shoelace*). Comparamos polígono contra polígono para que los dos se midan igual.

In [ ]:
def casco_convexo(puntos):
    pts = sorted(set(map(tuple, puntos.tolist())))

    def cruz(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    def media_cadena(secuencia):
        cadena = []
        for p in secuencia:
            while len(cadena) >= 2 and cruz(cadena[-2], cadena[-1], p) <= 0:
                cadena.pop()
            cadena.append(p)
        return cadena

    abajo, arriba = media_cadena(pts), media_cadena(reversed(pts))
    return np.array(abajo[:-1] + arriba[:-1])


def area_poligono(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


for r in regiones:
    r["casco"] = casco_convexo(r["contorno"])
    r["solidez"] = area_poligono(r["contorno"]) / area_poligono(r["casco"])
    r["convexidad"] = perimetro(r["casco"]) / r["perimetro"]
    print(f"Región {r['id']}: casco de {len(r['casco'])} vértices   "
          f"solidez = {r['solidez']:.3f}   convexidad = {r['convexidad']:.3f}")

El cuadrado y el triángulo dan ~1, como se espera. **¿Y el círculo con convexidad 0.95?** Su borde en píxeles es una escalerita: el contorno sube y baja en cada escalón, pero el casco pasa por encima de las puntas en línea recta. El casco mide menos que la escalerita, y la convexidad baja aunque la figura sí sea convexa. Es otra vez el **error de discretización**.

Para ver una diferencia de verdad dibujamos una **estrella** y una **L**, también con NumPy: para cada píxel trazamos un rayo horizontal y contamos cuántos lados del polígono cruza (**par = afuera, impar = adentro**). Es el *ray casting*, vectorizado por lado.

In [ ]:
def rellenar_poligono(forma, vertices):
    alto, ancho = forma
    yy, xx = np.mgrid[:alto, :ancho]
    dentro = np.zeros(forma, bool)
    v = np.asarray(vertices, float)
    for (x1, y1), (x2, y2) in zip(v, np.roll(v, -1, axis=0)):
        cruza = (y1 > yy) != (y2 > yy)
        with np.errstate(divide="ignore", invalid="ignore"):
            x_corte = x1 + (yy - y1) * (x2 - x1) / (y2 - y1)
        dentro ^= cruza & (xx < x_corte)
    return dentro


angulos = np.linspace(-np.pi / 2, 3 * np.pi / 2, 11)[:-1]
radios = np.where(np.arange(10) % 2 == 0, 120, 50)
estrella = np.stack([150 + radios * np.cos(angulos), 150 + radios * np.sin(angulos)], 1)
ele = [(380, 40), (450, 40), (450, 190), (560, 190), (560, 260), (380, 260)]
extra = rellenar_poligono((300, 600), estrella) | rellenar_poligono((300, 600), ele)

etiq_extra, n_extra = etiquetar(extra)
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(extra)
for i in range(1, n_extra + 1):
    c = trazar_contorno(etiq_extra == i)
    h = casco_convexo(c)
    sol = area_poligono(c) / area_poligono(h)
    conv = perimetro(h) / perimetro(c)
    ax.plot(*np.vstack([h, h[:1]]).T, "m", lw=2)
    cx, cy = c.mean(axis=0)
    ax.text(cx, cy + (0 if i == 1 else 30), f"solidez = {sol:.2f}\nconvexidad = {conv:.2f}",
            ha="center", va="center", color="w", fontsize=11, weight="bold",
            bbox=dict(facecolor="k", alpha=0.8))
ax.set_title("Magenta = casco convexo"); ax.axis("off")
plt.show()

## 10. Excentricidad: momentos de segundo orden

La **matriz de covarianza** de las coordenadas (x, y) de los píxeles describe cómo se reparte la masa de la figura:

$$ C = \begin{pmatrix} \sigma_{xx} & \sigma_{xy} \\ \sigma_{xy} & \sigma_{yy} \end{pmatrix} $$

Sus **eigenvalores** $\lambda_1 \ge \lambda_2$ son la varianza a lo largo del eje mayor y del menor, y sus **eigenvectores** la orientación. La elipse equivalente tiene ejes $4\sqrt{\lambda}$ y:

$$ excentricidad = \sqrt{1 - \lambda_2 / \lambda_1} $$

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(imagen)
t = np.linspace(0, 2 * np.pi, 100)
for r in regiones:
    C = np.cov(np.vstack([r["xs"], r["ys"]]), bias=True)
    lambdas, vectores = np.linalg.eigh(C)        # orden ascendente
    l2, l1 = lambdas
    r["excentricidad"] = np.sqrt(1 - l2 / l1)
    x0, y0, x1, y1 = r["bbox"]
    r["aspecto"] = (x1 - x0 + 1) / (y1 - y0 + 1)

    # elipse: círculo unitario escalado por los semiejes y rotado por los eigenvectores
    elipse = vectores @ np.diag(2 * np.sqrt(lambdas)) @ np.vstack([np.cos(t), np.sin(t)])
    cx, cy = r["centroide"]
    ax.plot(elipse[0] + cx, elipse[1] + cy, "k", lw=2)
    ax.text(cx, cy, f"exc = {r['excentricidad']:.2f}\naspecto = {r['aspecto']:.2f}",
            ha="center", va="center", color="w", fontsize=11, weight="bold")
ax.set_title("Elipse equivalente (eigenvectores de la covarianza)"); ax.axis("off")
plt.show()

## 11. La tabla, ahora sí con pandas

Aquí pandas sí ayuda: un `DataFrame` con una fila por figura, redondeo, y `apply` para clasificar cada fila.

In [ ]:
columnas = ["area", "perimetro", "circularidad", "extension", "solidez",
            "convexidad", "excentricidad", "aspecto"]
tabla = pd.DataFrame([{c: r[c] for c in columnas} for r in regiones],
                     index=pd.Index([r["id"] for r in regiones], name="región")).astype(float)


def clasificar(fila):
    if fila.circularidad > 0.85 and fila.excentricidad < 0.3:
        return "círculo"
    if fila.extension > 0.9:
        return "cuadrado" if 0.9 < fila.aspecto < 1.1 else "rectángulo"
    if 0.4 < fila.extension < 0.6 and fila.solidez > 0.95:
        return "triángulo"
    return "?"


tabla["forma"] = tabla.apply(clasificar, axis=1)
tabla.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(imagen)
for r in regiones:
    ax.text(*r["centroide"], tabla.loc[r["id"], "forma"], ha="center", va="center",
            color="w", fontsize=14, weight="bold", bbox=dict(facecolor="k", alpha=0.5))
ax.set_title("Clasificación"); ax.axis("off")
plt.show()

## 12. Comprobación (opcional): ¿coincide con scikit-image?

Si tienes `scikit-image` (en Colab ya viene), comparamos contra `regionprops`. El área y la extensión deben ser idénticas. El perímetro difiere un poco porque scikit-image usa otra forma de estimarlo, y por eso la circularidad cambia en el segundo decimal.

In [ ]:
try:
    from skimage import measure
except ImportError:
    print("No está scikit-image; sáltate esta celda.")
else:
    comparacion = pd.DataFrame([{
        "área (nuestra)": r["area"], "área (skimage)": p.area,
        "perímetro (nuestro)": r["perimetro"], "perímetro (skimage)": p.perimeter,
        "extensión (nuestra)": r["extension"], "extensión (skimage)": p.extent,
        "excentricidad (nuestra)": r["excentricidad"], "excentricidad (skimage)": p.eccentricity,
    } for r, p in zip(regiones, measure.regionprops(etiquetas))], index=tabla.index)
    display(comparacion.round(3).T)

## Ejercicios

1. **Velocidad.** Mide con `%time` cuánto tarda `etiquetar` contra `skimage.measure.label`. ¿Por qué tanta diferencia?
2. **Conectividad.** Cambia `VECINOS_8` por solo 4 vecinos (arriba, abajo, izquierda, derecha). Dibuja dos cuadros que se toquen solo en una esquina: ¿cuántas regiones salen con 4 y con 8?
3. **Perímetro.** Cuenta el perímetro como el número de puntos del contorno (todos los pasos valen 1). ¿Qué le pasa a la circularidad del círculo? ¿Por qué el √2 importa?
4. **Kernel.** Cambia el kernel por una cruz 3×3 y por un cuadrado 7×7. ¿Cómo cambian la erosión y la dilatación?